# Geosteering AI v2.40 — Benchmark tf.data + Mixed Precision

**Template**: `benchmark_tfdata_mp16.ipynb` | **Sprint**: v2.40

Mede throughput de treinamento (`samples/sec`) em 4 configurações:

| Config | mp16 | XLA | tf.data tuning |
|:---|:---:|:---:|:---:|
| C1 baseline | ❌ | ❌ | default |
| C2 mp16 | ✅ | ❌ | default |
| C3 mp16+XLA | ✅ | ✅ | default |
| C4 mp16+XLA+tf.data | ✅ | ✅ | shuffle=4096, parallel=8 |

Cada config: **5 runs**, mediana reportada. Stdev > 5% → reject + retry.

**Saída**: `docs/perf_baselines/v2.40_tf_training_{device}.json` (commitado pelo usuário).

## Cell 1 — Configuração

In [ ]:
GIT_TAG = "v2.40"
RUNTIME_DEVICE = "T4"  # editar para A100 se disponível
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/v2.40/perf_baselines/"

N_RUNS = 5  # mediana de 5 runs
N_EPOCHS_BENCH = 3  # poucas épocas para medir throughput (não converger)
BATCH_SIZE = 64
SEED = 42

print(f"✓ Benchmark v2.40 em {RUNTIME_DEVICE}: {N_RUNS} runs x 4 configs")

## Cell 2 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone --depth 1 --branch {GIT_TAG} https://github.com/daniel-guitarplayer-8/geosteering-ai.git
%cd geosteering-ai
!pip install -q -e ".[all]"

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
assert len(gpus) > 0
print(f"✓ GPU: {gpus[0].name}")

## Cell 3 — Função de Benchmark

In [ ]:
import time
import numpy as np
from geosteering_ai.config import PipelineConfig
from geosteering_ai.data.pipeline import DataPipeline, PreparedData
from geosteering_ai.models.registry import build_model_with_mp_policy
from geosteering_ai.training.loop import setup_mixed_precision_policy

def bench_config(config_label, **config_kwargs):
    """Mede samples/sec em config."""
    config = PipelineConfig(
        model_type="ResNet_18",
        sequence_length=10,  # mínimo p/ benchmark
        batch_size=BATCH_SIZE,
        epochs=N_EPOCHS_BENCH,
        seed=SEED,
        **config_kwargs,
    )
    setup_mixed_precision_policy(config)
    
    # Dataset sintético (1024 samples, 10 seq, 5 feat, 2 targets)
    n = 1024
    x = np.random.default_rng(SEED).normal(size=(n, 10, 5)).astype(np.float32)
    y = np.random.default_rng(SEED+1).normal(size=(n, 10, 2)).astype(np.float32)
    
    pipeline = DataPipeline(config)
    prepared = PreparedData(x_train=x, y_train=y, x_val=x[:128], y_val=y[:128],
                            x_test=x[:128], y_test=y[:128])
    train_ds = pipeline.build_tf_dataset(prepared, "train", noise_level_var=None)
    val_ds = pipeline.build_tf_dataset(prepared, "val", noise_level_var=None)
    
    model = build_model_with_mp_policy(config)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse",
        jit_compile=config.use_xla,
    )
    
    # Warmup 1 epoch (descartar)
    model.fit(train_ds, epochs=1, verbose=0)
    
    # Bench N_RUNS
    throughputs = []
    for run_i in range(N_RUNS):
        t0 = time.perf_counter()
        model.fit(train_ds, epochs=N_EPOCHS_BENCH, verbose=0)
        elapsed = time.perf_counter() - t0
        thru = (n * N_EPOCHS_BENCH) / elapsed
        throughputs.append(thru)
        print(f"  [{config_label}] run {run_i+1}/{N_RUNS}: {thru:.1f} samples/sec")
    
    median = float(np.median(throughputs))
    stdev = float(np.std(throughputs))
    stdev_pct = (stdev / median) * 100 if median > 0 else 0
    return {
        "label": config_label,
        "throughputs": throughputs,
        "median_samples_per_sec": median,
        "stdev_samples_per_sec": stdev,
        "stdev_pct": stdev_pct,
        "valid": stdev_pct < 5.0,
    }

print("✓ bench_config definida")

## Cell 4 — Executar 4 Configurações

In [ ]:
all_results = {}

print("=== C1: baseline (fp32, sem XLA, tf.data default) ===")
all_results["C1_baseline_fp32"] = bench_config(
    "C1_baseline",
    use_mixed_precision=False,
    use_xla=False,
)

print("\n=== C2: +mp16 ===")
all_results["C2_mp16"] = bench_config(
    "C2_mp16",
    use_mixed_precision=True,
    use_xla=False,
)

print("\n=== C3: +mp16+XLA ===")
all_results["C3_mp16_xla"] = bench_config(
    "C3_mp16_xla",
    use_mixed_precision=True,
    use_xla=True,
)

print("\n=== C4: +mp16+XLA+tf.data tuned ===")
all_results["C4_mp16_xla_tfdata"] = bench_config(
    "C4_mp16_xla_tfdata",
    use_mixed_precision=True,
    use_xla=True,
    tf_shuffle_buffer_size=4096,
    tf_num_parallel_calls=8,
    tf_prefetch_buffer_size=4,
)

## Cell 5 — Análise + Export JSON

In [ ]:
import os
import json
import datetime

baseline = all_results["C1_baseline_fp32"]["median_samples_per_sec"]

print("\n" + "=" * 70)
print(f"BENCHMARK v2.40 — {RUNTIME_DEVICE}")
print("=" * 70)
print(f"{'Config':<25} {'samples/sec':>14} {'speedup':>10} {'stdev %':>10}")
print("-" * 70)
for label, r in all_results.items():
    speedup = r["median_samples_per_sec"] / baseline if baseline > 0 else 0
    print(f"{label:<25} {r['median_samples_per_sec']:>14.1f} {speedup:>9.2f}x {r['stdev_pct']:>9.2f}%")
print("=" * 70)

# Ganho mp16: gate de aceitação Sprint v2.40 (≥15% em T4)
c2_speedup = all_results["C2_mp16"]["median_samples_per_sec"] / baseline
gate_pass = c2_speedup >= 1.15
print(f"\nGate v2.40 (mp16 ≥ 1.15x baseline): {'✓ PASS' if gate_pass else '✗ FAIL'} ({c2_speedup:.2f}x)")

# Export
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DRIVE_DIR, f"v2.40_tf_training_{RUNTIME_DEVICE}_{ts}.json")

with open(out_path, "w") as f:
    json.dump({
        "template": "benchmark_tfdata_mp16",
        "git_tag": GIT_TAG,
        "runtime": f"colab_pro_plus_{RUNTIME_DEVICE}",
        "timestamp_utc": ts,
        "n_runs_per_config": N_RUNS,
        "n_epochs_per_run": N_EPOCHS_BENCH,
        "batch_size": BATCH_SIZE,
        "results": all_results,
        "gate_v2.40_mp16_speedup_min": 1.15,
        "gate_passed": gate_pass,
        "c2_speedup_vs_baseline": c2_speedup,
    }, f, indent=2)

print(f"\n✓ JSON exportado: {out_path}")
print("  Próximo passo: commitar em docs/perf_baselines/v2.40_tf_training_T4.json")